# TMTR — TimeGAN baseline

## FTS-Diffusion Paper, Section 5.3 / Fig. 6(a), with TimeGAN as the generator

This notebook replicates the **TMTR** (Train on Mixture, Test on Real) experiment using
**TimeGAN (Yoon, Jarrett, van der Schaar — NeurIPS 2019)** as the synthetic
data generator, instead of FTS-Diffusion. Same downstream LSTM, same
evaluation protocol.

### TimeGAN setup (paper-faithful, from FTS-Diffusion paper Appendix E.2)
- **Univariate**: daily returns of closing prices (`r_t = (p_t − p_{t-1}) / p_{t-1}`)
- **Window length**: **21** (matches `l_max` of SISC in FTS-Diffusion)
- **Stitching**: synthetic price series of length `T` is built by generating
  `ceil(T/21)` independent 21-day return windows, concatenating them, then
  applying `prices = p_init × cumprod(1 + returns)`.

### TimeGAN protocols
TimeGAN does NOT have a Markov chain (each window comes from an independent
`Z`), so the FTS-Diffusion protocols (`authors / split / random_init`) collapse
into a single one: `timegan_default`. Comparison curves will be:

> FTS-Diffusion (`authors`, `split`, …) **vs** TimeGAN (`timegan_default`)

### Persistent storage on Drive (parallel to FTS-Diffusion)
```
/content/drive/MyDrive/fts_diffusion/
├── architectures/{asset}/timegan/    # TimeGAN model.pt + args.pickle
├── synthetic/{asset}/timegan/        # Generated synthetic series per run
├── results/tmtr/{asset}/timegan/   # MAPE CSVs + summary
└── figures/tmtr/{asset}/timegan/   # Final plots
```

### LSTM hyperparameters (matches the FTS-Diffusion notebook for fair compare)
| LSTM hidden_dim | 64 |
| LSTM layers | 2 |
| Window size | 64 |
| Loss | MSE |
| Optimizer | Adam, lr=1e-2 |
| Epochs | 200 |


## 1. Environment Setup


In [ ]:
import os, sys, subprocess, time, json, shutil, pickle
from pathlib import Path
from types import SimpleNamespace

IS_COLAB = 'google.colab' in sys.modules
print(f"Running on Colab: {IS_COLAB}")

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # 1) Our repo
    REPO_URL = 'https://github.com/thaidinger/ml-in-fcs.git'
    BRANCH = 'deli_work'
    CLONE_DIR = '/content/ml-in-fcs'
    if not os.path.exists(CLONE_DIR):
        subprocess.check_call(['git', 'clone', '-b', BRANCH, REPO_URL, CLONE_DIR])
    else:
        subprocess.check_call(['git', '-C', CLONE_DIR, 'pull'])
    REPO_ROOT = f'{CLONE_DIR}/fts-diffusion-ref'

    # 2) TimeGAN repo
    TIMEGAN_DIR = '/content/timegan-pytorch'
    if not os.path.exists(TIMEGAN_DIR):
        subprocess.check_call(['git', 'clone', '--depth', '1',
                               'https://github.com/birdx0810/timegan-pytorch.git', TIMEGAN_DIR])
    else:
        print(f"  TimeGAN repo already at {TIMEGAN_DIR}")

    PERSIST_ROOT = '/content/drive/MyDrive/fts_diffusion'
else:
    REPO_ROOT    = '/home/deli/ETH_projects/ml-in-fcs/fts-diffusion-ref'
    TIMEGAN_DIR  = '/tmp/timegan-pytorch'
    if not os.path.exists(TIMEGAN_DIR):
        subprocess.check_call(['git', 'clone', '--depth', '1',
                               'https://github.com/birdx0810/timegan-pytorch.git', TIMEGAN_DIR])
    PERSIST_ROOT = '/home/deli/ETH_projects/ml-in-fcs/fts_diffusion_run'

os.makedirs(PERSIST_ROOT, exist_ok=True)
for sub in ['architectures', 'synthetic', 'results', 'figures']:
    os.makedirs(f'{PERSIST_ROOT}/{sub}', exist_ok=True)

# Make BOTH repos importable: ours (for downstream/data utilities) + TimeGAN (for the model)
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, TIMEGAN_DIR)
os.chdir(REPO_ROOT)
for d in ['data', 'res', 'trained_models', 'figs']:
    os.makedirs(d, exist_ok=True)

print(f"Working dir:    {os.getcwd()}")
print(f"TimeGAN dir:    {TIMEGAN_DIR}")
print(f"Persistent dir: {PERSIST_ROOT}")


In [ ]:
# Install dependencies
for pkg in ['numpy', 'pandas', 'matplotlib', 'scipy', 'scikit-learn', 'torch', 'tqdm', 'tslearn', 'tensorboard', 'yfinance']:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print("✓ Deps ready")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_absolute_percentage_error as MAPE
from sklearn.preprocessing import MinMaxScaler
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Our repo modules (data + downstream LSTM utilities — same as FTS-Diffusion notebooks)
# These live in fts-diffusion-ref/{utils,experiments} — distinct top-level packages,
# so they don't collide with TimeGAN's `models` package.
from utils.load_data import get_fts, load_actual_fts
from experiments.utils_downstream import (
    get_downstream_data, init_first_segment, init_tatr_set,
    Timeseries2Dataset_Downstream, create_syn_dataset,
    concat_datasets_downstream, copy_dataset_downstream)
from experiments.predictor_lstm import LSTMPredictor, set_loss_fn

# === Import TimeGAN's `models` package ===
# Both repos have a top-level `models/` package. Python caches the first one it
# imports, so if you ran an FTS-Diffusion notebook in this kernel earlier the
# `models` cache points at fts-diffusion-ref/models/ and `from models.timegan`
# fails. Wipe any cached `models*` modules and re-import from TIMEGAN_DIR (which
# is at sys.path[0]).
import importlib
for m in [k for k in list(sys.modules) if k == 'models' or k.startswith('models.')]:
    del sys.modules[m]
from models.timegan import TimeGAN
from models.utils import timegan_trainer, timegan_generator

print("✓ Imports ready (FTS utilities + TimeGAN)")


## 2. Configuration


In [ ]:
# ============================================================================
# === EXPERIMENT CONFIG ===
# ============================================================================

ASSET          = 'sp500'      # 'sp500' | 'goog' | 'zcf'
EXPERIMENT     = 'tmtr'
GENERATOR      = 'timegan'    # fixed for this notebook

# Per-asset metadata
ASSETS_CONFIG = {
    'sp500':  {'ticker': '^GSPC', 'start': '1980-01-01', 'end': '2020-01-01', 'pretty': 'S&P 500'},
    'goog':   {'ticker': 'GOOG',  'start': '2005-01-01', 'end': '2020-01-01', 'pretty': 'GOOG'},
    'zcf':    {'ticker': 'ZC=F',  'start': '2001-01-01', 'end': '2020-01-01', 'pretty': 'ZC=F (Corn futures)'},
}
assert ASSET in ASSETS_CONFIG, f"Unknown asset {ASSET}"

DATANAME       = ASSET
TICKER         = ASSETS_CONFIG[ASSET]['ticker']
START_DATE     = ASSETS_CONFIG[ASSET]['start']
END_DATE       = ASSETS_CONFIG[ASSET]['end']
PRETTY_NAME    = ASSETS_CONFIG[ASSET]['pretty']

# === Persistent paths ===
ARCH_DIR    = f'{PERSIST_ROOT}/architectures/{ASSET}/{GENERATOR}'
SYN_DIR     = f'{PERSIST_ROOT}/synthetic/{ASSET}/{GENERATOR}'
RES_DIR     = f'{PERSIST_ROOT}/results/{EXPERIMENT}/{ASSET}/{GENERATOR}'
FIG_DIR     = f'{PERSIST_ROOT}/figures/{EXPERIMENT}/{ASSET}/{GENERATOR}'
for d in [ARCH_DIR, SYN_DIR, RES_DIR, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

# === TimeGAN hyperparameters ===
TG_WINDOW       = 21         # paper Appendix E.2: matches l_max of SISC
TG_FEATURE_DIM  = 1          # univariate returns
TG_HIDDEN_DIM   = 24         # birdx0810 default
TG_NUM_LAYERS   = 3
TG_BATCH_SIZE   = 128
TG_LR           = 1e-3
TG_DIS_THRESH   = 0.15
# Reduced from default (600/600/600) to fit Colab; bump if quality is poor.
TG_EMB_EPOCHS   = 200
TG_SUP_EPOCHS   = 200
TG_GAN_EPOCHS   = 200
TG_SEED         = 42

# === Downstream LSTM hyperparameters (shared with FTS-Diffusion) ===
WINDOW_SIZE    = 64
LSTM_LAYERS    = 2
AHEAD          = 1
LR             = 1e-2
DATATYPE       = 'prices'

# === TMTR-specific ===
N_RUNS         = 20

# MIX_LENGTH is per-asset because get_downstream_data() returns only the SISC
# test split (~20% of total history). For S&P 500 (downstream ~2000d) the paper's
# 1260d (5y) fits; for GOOG/ZC=F the downstream is shorter and we need to shrink.
# Auto-shrink fallback in setup cell handles cases where even these are too big.
MIX_LENGTH_PER_ASSET = {
    'sp500':    252 * 5,    # 1260d = 5y (paper-faithful, downstream ~2000d)
    'goog':     252 * 2,    #  504d = 2y (downstream only ~780d for GOOG)
    'zcf':      252 * 3,    #  756d = 3y (downstream ~1000d for ZC=F)
}
MIX_LENGTH     = MIX_LENGTH_PER_ASSET.get(ASSET, 252 * 5)

N_PROPORTIONS  = 10
PROPORTIONS    = list(range(0, 101, int(100 / N_PROPORTIONS)))   # [0, 10, ..., 100]

# === LSTM CONFIG (paper-faithful TMTR) ===
LSTM_HIDDEN    = 64
N_EPOCHS       = 200
LSTM_LOSS      = 'mse'


print(f"Configuration:")
print(f"  ASSET         = {ASSET}  ({PRETTY_NAME})")
print(f"  EXPERIMENT    = {EXPERIMENT.upper()}")
print(f"  GENERATOR     = {GENERATOR}")
print(f"  TG window     = {TG_WINDOW} days, hidden={TG_HIDDEN_DIM}, layers={TG_NUM_LAYERS}")
print(f"  TG epochs     = emb:{TG_EMB_EPOCHS} + sup:{TG_SUP_EPOCHS} + gan:{TG_GAN_EPOCHS}")
print(f"  MIX_LENGTH    = {MIX_LENGTH} days ({MIX_LENGTH/252:.1f} years)  [per-asset default]")
print()
print(f"Drive paths:")
print(f"  ARCH_DIR = {ARCH_DIR}")
print(f"  SYN_DIR  = {SYN_DIR}")
print(f"  RES_DIR  = {RES_DIR}")
print(f"  FIG_DIR  = {FIG_DIR}")


## 3. Phase 1 — Load asset & build sliding return windows


In [ ]:
# Step 1: Make sure asset CSV is available (reuse FTS-Diffusion data dir)
ts_path = f'data/{DATANAME}_timeseries.csv'

# Reuse from Drive if FTS-Diffusion already downloaded it
fts_arch_data = f'{PERSIST_ROOT}/architectures/{ASSET}'
for k_dir in os.listdir(fts_arch_data) if os.path.exists(fts_arch_data) else []:
    candidate = f'{fts_arch_data}/{k_dir}/data/{DATANAME}_timeseries.csv'
    if os.path.exists(candidate) and not os.path.exists(ts_path):
        shutil.copy(candidate, ts_path)
        print(f"✓ Reused FTS-Diffusion data: {candidate} → {ts_path}")
        break

if not os.path.exists(ts_path):
    print(f"Downloading {TICKER} from {START_DATE} to {END_DATE}...")
    get_fts(ticker=TICKER, fts_name=DATANAME, start_date=START_DATE, end_date=END_DATE)
else:
    print(f"✓ Data already at {ts_path}")

prices_full = load_actual_fts(DATANAME).squeeze()
prices_full = np.asarray(prices_full, dtype=np.float64)
print(f"{PRETTY_NAME} series: {len(prices_full)} points, range [{prices_full.min():.4f}, {prices_full.max():.4f}]")


In [ ]:
# Step 2: Compute returns and build sliding windows of length TG_WINDOW
returns_full = (prices_full[1:] - prices_full[:-1]) / prices_full[:-1]
print(f"Returns: {returns_full.shape}, mean={returns_full.mean():.5f}, std={returns_full.std():.5f}")

# Use authors' downstream split: TimeGAN sees only the LSTM-train portion of the data,
# matching what FTS-Diffusion sees (no leakage of LSTM test period into TimeGAN training).
downstream_ts, segments_test, labels_test, lengths_test = get_downstream_data()
INIT_FRACTION = 0.625        # = 5/8, same as paper for S&P 500
total_len = len(downstream_ts)
init_period = int(total_len * INIT_FRACTION)
init_period = max(init_period, 252)

train_prices = downstream_ts[:init_period]
train_returns = (train_prices[1:] - train_prices[:-1]) / train_prices[:-1]
print(f"\nTimeGAN training window:")
print(f"  Train prices:  {len(train_prices)} days ({len(train_prices)/252:.2f} years)")
print(f"  Train returns: {len(train_returns)} days")

# Sliding windows of length TG_WINDOW
def sliding_windows_returns(returns_1d, window):
    n = len(returns_1d) - window + 1
    if n <= 0:
        raise ValueError(f"Returns too short ({len(returns_1d)}) for window {window}")
    out = np.zeros((n, window, 1), dtype=np.float32)
    for i in range(n):
        out[i, :, 0] = returns_1d[i:i+window]
    return out

X_train = sliding_windows_returns(train_returns, TG_WINDOW)
T_train = np.full(len(X_train), TG_WINDOW, dtype=np.int64)
print(f"\nTimeGAN training tensor: X={X_train.shape}, T={T_train.shape}")

# Standardize returns (TimeGAN expects roughly unit-scale inputs; we will invert later)
ret_mean = train_returns.mean()
ret_std  = train_returns.std() + 1e-12
X_train_scaled = (X_train - ret_mean) / ret_std
print(f"Scaling: mean={ret_mean:.6f}, std={ret_std:.6f}")


## 4. Phase 2 — Train TimeGAN (once per asset)

Expensive step (~30-60 min on A100 with reduced epochs). Checkpoint persists
on Drive at `ARCH_DIR/model.pt` and `ARCH_DIR/args.pickle`.


In [ ]:
# Build the args namespace expected by birdx0810/timegan-pytorch
def make_timegan_args():
    args = SimpleNamespace()
    args.device         = device
    args.exp            = f'{ASSET}_timegan'
    args.is_train       = True
    args.seed           = TG_SEED
    args.max_seq_len    = TG_WINDOW
    args.feature_dim    = TG_FEATURE_DIM
    args.Z_dim          = TG_FEATURE_DIM
    args.padding_value  = 0.0
    args.hidden_dim     = TG_HIDDEN_DIM
    args.num_layers     = TG_NUM_LAYERS
    args.batch_size     = TG_BATCH_SIZE
    args.learning_rate  = TG_LR
    args.dis_thresh     = TG_DIS_THRESH
    args.optimizer      = 'adam'
    args.emb_epochs     = TG_EMB_EPOCHS
    args.sup_epochs     = TG_SUP_EPOCHS
    args.gan_epochs     = TG_GAN_EPOCHS
    args.model_path     = ARCH_DIR
    return args


args = make_timegan_args()
model_pt    = f'{ARCH_DIR}/model.pt'
args_pickle = f'{ARCH_DIR}/args.pickle'

if os.path.exists(model_pt) and os.path.exists(args_pickle):
    print(f"✓ TimeGAN checkpoint found at {ARCH_DIR} — skipping training.")
    # Re-load args (preserves the original training seed/hyperparams)
    with open(args_pickle, 'rb') as f:
        args_loaded = torch.load(f)
    # But override dynamic fields (device may differ across runs)
    args_loaded.device = device
    args_loaded.model_path = ARCH_DIR
    args = args_loaded
    print(f"  Loaded args: hidden={args.hidden_dim}, layers={args.num_layers}, "
          f"epochs=({args.emb_epochs},{args.sup_epochs},{args.gan_epochs})")
    model = TimeGAN(args).to(device)
    model.load_state_dict(torch.load(model_pt, map_location=device))
    model.eval()
else:
    print(f"Training TimeGAN from scratch on {PRETTY_NAME} returns...")
    print(f"  Total epochs: {TG_EMB_EPOCHS + TG_SUP_EPOCHS + TG_GAN_EPOCHS}, "
          f"batch={TG_BATCH_SIZE}, hidden={TG_HIDDEN_DIM}")
    print(f"  Estimated: ~30-60 min on A100")
    t0 = time.time()
    model = TimeGAN(args).to(device)
    timegan_trainer(model, X_train_scaled, T_train, args)
    elapsed = (time.time() - t0) / 60
    print(f"✓ TimeGAN trained in {elapsed:.1f} min  (saved to {ARCH_DIR})")
    model.eval()


## 5. Phase 3 — Synthetic data generator wrapper

Stitches independent 21-day return windows from TimeGAN, then converts to
prices via `cumprod(1 + returns)`. Mirrors the role of
`generate_timeseries_ftsdiffusion(T, init_state, init_segment)` in the
FTS-Diffusion notebooks, so the downstream TATR/TMTR loop is identical.


In [ ]:
@torch.no_grad()
def generate_returns_timegan(n_windows, model, args, ret_mean, ret_std, device):
    """Generate n_windows independent 21-day return windows from TimeGAN.

    Returns: 1D numpy of length n_windows * args.max_seq_len (un-scaled returns).
    """
    Z = torch.rand((n_windows, args.max_seq_len, args.Z_dim), device=device)
    T_gen = torch.full((n_windows,), args.max_seq_len, dtype=torch.long)
    model.eval()
    out = model(X=None, T=T_gen, Z=Z, obj='inference')
    if out.is_cuda:
        out = out.cpu()
    out_np = out.numpy()                                 # (n_windows, max_seq_len, 1)
    # Invert the standardization
    out_np = out_np * ret_std + ret_mean
    return out_np.reshape(-1)                            # (n_windows * max_seq_len,)


def generate_timeseries_timegan(T, model, args, p_init, ret_mean, ret_std,
                                 device, seed=None):
    """Generate T days of synthetic prices using TimeGAN.

    1. Sample ceil(T / window) independent 21-day return windows.
    2. Concatenate, trim to T returns.
    3. prices[0] = p_init, prices[t] = p_init * cumprod(1 + returns)[t-1].
    Returns 1D numpy float32 of length T.
    """
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)

    window = args.max_seq_len
    n_windows = (T + window - 1) // window               # ceil
    returns = generate_returns_timegan(n_windows, model, args, ret_mean, ret_std, device)
    returns = returns[:T - 1]                            # T prices need T-1 returns

    # Cap returns to avoid pathological compounding (TimeGAN can produce extreme tails)
    returns = np.clip(returns, -0.50, 0.50)

    prices = np.empty(T, dtype=np.float32)
    prices[0] = p_init
    if T > 1:
        prices[1:] = p_init * np.cumprod(1.0 + returns)
    return prices


print("✓ TimeGAN sampling wrapper ready")

# Sanity check: generate one short series and plot
syn_demo = generate_timeseries_timegan(252, model, args,
                                       p_init=float(downstream_ts[0]),
                                       ret_mean=ret_mean, ret_std=ret_std,
                                       device=device, seed=0)
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].plot(downstream_ts[:252], label='Real (first 252d)')
axes[0].plot(syn_demo, label='TimeGAN synth (252d)', alpha=0.7)
axes[0].set_title('Demo: 252-day real vs synthetic prices')
axes[0].legend()
axes[1].plot(np.diff(np.log(downstream_ts[:252])), label='Real log-returns', alpha=0.7)
axes[1].plot(np.diff(np.log(syn_demo)), label='Synth log-returns', alpha=0.7)
axes[1].set_title('Log-returns')
axes[1].legend()
plt.tight_layout(); plt.show()


## 6. Phase 4 — TMTR Loop


In [ ]:
# === Setup TMTR datasets ===
#
# get_downstream_data() returns only the SISC test split (~20% of total history),
# not the full asset series. So:
#   - S&P 500 (~10k total) → downstream ~2000d → MIX_LENGTH=1260 fits (paper-faithful)
#   - GOOG    (~3.8k total) → downstream  ~780d → must shrink MIX_LENGTH
#   - ZC=F    (~5k total)  → downstream ~1000d → must shrink MIX_LENGTH
# Auto-shrinks MIX_LENGTH if needed and prints a clear warning.
#
# Additionally, each non-edge proportion `p` produces chunks of size
#   syn_len = MIX_LENGTH * p / 100,    real_len = MIX_LENGTH - syn_len
# and create_syn_dataset uses a sliding window of WINDOW_SIZE. So we need
# both chunks ≥ WINDOW_SIZE+1 days for unfold() to work. Auto-filter PROPORTIONS
# to drop the unsafe ones.

def adapt_proportions(proportions, mix_length, window_size):
    """Drop proportions that would produce a chunk smaller than window_size+1 days."""
    min_chunk = window_size + 1
    keep = []
    for p in proportions:
        syn_len  = int(mix_length * p / 100)
        real_len = mix_length - syn_len
        if p == 0:
            if real_len >= min_chunk: keep.append(p)
        elif p == 100:
            if syn_len >= min_chunk: keep.append(p)
        else:
            if syn_len >= min_chunk and real_len >= min_chunk:
                keep.append(p)
    return keep


def setup_dowmstream_tmtr_adaptive(window_size, mix_length=MIX_LENGTH, proportions=None):
    """Returns (effective_mix_length, effective_proportions, real_ts, test_ts,
                test_dataset, scaler, p_anchor)."""
    if proportions is None:
        proportions = PROPORTIONS

    downstream_ts, segments_test, labels_test, lengths_test = get_downstream_data()
    total_len = len(downstream_ts)

    # Need ≥ window_size+1 days for ≥ 1 LSTM test window; want ≥ 252d (1y) for meaningful eval.
    min_test = max(window_size + 1, 252)

    if total_len < mix_length + min_test:
        new_mix_length = max(int(total_len * 0.6), 252)
        if new_mix_length + min_test > total_len:
            raise ValueError(
                f"Asset {ASSET} too short for TMTR even after auto-shrink: "
                f"downstream={total_len}d, need ≥ {252 + min_test}d.")
        print(f"⚠ Asset {ASSET} downstream is only {total_len}d.")
        print(f"  Auto-shrinking MIX_LENGTH: {mix_length} → {new_mix_length} days "
              f"(keeping ≥ {min_test}d as test set).")
        mix_length = new_mix_length

    # Filter PROPORTIONS to those that yield chunks ≥ window_size+1 on both sides
    eff_proportions = adapt_proportions(proportions, mix_length, window_size)
    if len(eff_proportions) < len(proportions):
        dropped = sorted(set(proportions) - set(eff_proportions))
        print(f"⚠ Dropping {len(dropped)} proportion(s) too small for WINDOW_SIZE={window_size}: {dropped}")
        print(f"  (chunk would be < {window_size + 1} days — sliding window can't fit)")
        print(f"  Effective sweep: {eff_proportions}")

    real_ts = downstream_ts[:mix_length]
    test_ts = downstream_ts[mix_length:]
    _, scaler = Timeseries2Dataset_Downstream(test_ts, window_size)
    test_dataset = Timeseries2Dataset_Downstream(test_ts, window_size, scaler)

    print(f"TMTR setup for {ASSET} ({PRETTY_NAME}) — TimeGAN baseline:")
    print(f"  Total downstream:         {total_len} days ({total_len/252:.2f} years)")
    print(f"  MIX_LENGTH (real anchor): {mix_length} days ({mix_length/252:.2f} years)")
    print(f"  Test set (held out):      {len(test_ts)} days ({len(test_ts)/252:.2f} years)")
    print(f"  test_dataset shape:       {test_dataset.shape}")
    print(f"  PROPORTIONS sweep:        {eff_proportions}  ({len(eff_proportions)} points)")
    return mix_length, eff_proportions, real_ts, test_ts, test_dataset, scaler, downstream_ts[0]


(MIX_LENGTH, PROPORTIONS, real_ts, test_ts, test_dataset, scaler, p_anchor) = \
    setup_dowmstream_tmtr_adaptive(WINDOW_SIZE, MIX_LENGTH, PROPORTIONS)
# ^ MIX_LENGTH and PROPORTIONS are rebound here so downstream cells use the effective values.


In [ ]:
# === LSTM training & evaluation (paper-faithful TMTR) ===
def train_lstm_full_batch(dataset, n_epochs=N_EPOCHS, hidden_dim=LSTM_HIDDEN,
                          n_layers=LSTM_LAYERS, output_dim=AHEAD, criterion_str=LSTM_LOSS,
                          lr=LR, seed=0):
    torch.manual_seed(seed)
    model_lstm = LSTMPredictor(input_dim=1, hidden_dim=hidden_dim, output_dim=output_dim,
                               n_layers=n_layers, device=device).to(device)
    criterion = set_loss_fn(criterion_str)
    optimizer = optim.Adam(model_lstm.parameters(), lr=lr)
    X = dataset[:, :-output_dim].unsqueeze(-1).to(device)
    y = dataset[:, -output_dim:].to(device)
    for _ in range(n_epochs):
        model_lstm.train()
        y_pred = model_lstm(X)
        loss = criterion(y_pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return model_lstm, float(loss.item())


def evaluate_lstm(model_lstm, test_dataset, scaler, output_dim=AHEAD):
    model_lstm.eval()
    with torch.no_grad():
        X = test_dataset[:, :-output_dim].unsqueeze(-1).to(device)
        y_true = test_dataset[:, -output_dim:].numpy()
        y_pred = model_lstm(X).cpu().numpy()
        return float(MAPE(scaler.inverse_transform(y_true), scaler.inverse_transform(y_pred)))


def build_mix_dataset(real_ts, syn_ts, real_len, syn_len, scaler):
    """Build the mixture training dataset of total length MIX_LENGTH.

    Mirrors authors' create_mixture_dataset: take the first `real_len` days of real
    and the first `syn_len` days of synthetic, concatenate as TensorDatasets.

    Edge cases:
      - syn_len == 0:                 all-real (real-only baseline)
      - real_len == 0:                all-synthetic
      - one side < WINDOW_SIZE+1:     too short to unfold any LSTM window; we
        fall back to using only the longer side (the proportion effectively
        snaps to 0% or 100% near the boundary). This kicks in for short
        MIX_LENGTH on small assets (e.g. GOOG with MIX_LENGTH=504, prop=10%
        gives syn_len=50 < 64=WINDOW_SIZE).
    """
    real_ok = real_len >= WINDOW_SIZE + 1
    syn_ok  = syn_len  >= WINDOW_SIZE + 1
    if not real_ok and not syn_ok:
        # Should be impossible: MIX_LENGTH >= WINDOW_SIZE+1 must hold.
        raise ValueError(
            f"Both chunks too short for window={WINDOW_SIZE}: "
            f"real_len={real_len}, syn_len={syn_len}, MIX_LENGTH={MIX_LENGTH}")
    if syn_len == 0 or not syn_ok:
        if not syn_ok and syn_len > 0:
            print(f"    [build_mix] syn chunk {syn_len}d < window {WINDOW_SIZE}+1 - using real-only")
        return create_syn_dataset(real_ts[:real_len], WINDOW_SIZE, scaler, DATATYPE)
    if real_len == 0 or not real_ok:
        if not real_ok and real_len > 0:
            print(f"    [build_mix] real chunk {real_len}d < window {WINDOW_SIZE}+1 - using synth-only")
        return create_syn_dataset(syn_ts[:syn_len], WINDOW_SIZE, scaler, DATATYPE)
    # Both sides large enough -> standard mix
    real_chunk = real_ts[:real_len]
    syn_chunk  = syn_ts[:syn_len]
    real_ds = create_syn_dataset(real_chunk, WINDOW_SIZE, scaler, DATATYPE)
    syn_ds  = create_syn_dataset(syn_chunk,  WINDOW_SIZE, scaler, DATATYPE)
    return concat_datasets_downstream(real_ds, syn_ds)

print("✓ Train/eval/mix functions ready")


In [ ]:
# === TMTR run for a single seed (TimeGAN generator) ===

def run_single_tmtr_timegan(run_idx, proportions, seed_base=42):
    """One TMTR replicate using TimeGAN as the synthetic generator."""
    run_csv = f'{RES_DIR}/run_{run_idx:02d}.csv'
    results = {}
    if os.path.exists(run_csv):
        df_prev = pd.read_csv(run_csv)
        results = dict(zip(df_prev['proportion_pct'].astype(int), df_prev['mape'].astype(float)))
        if set(proportions).issubset(set(results.keys())):
            print(f"  ✓ Run {run_idx}: all proportions already done.")
            return {p: results[p] for p in proportions}

    syn_path = f'{SYN_DIR}/run_{run_idx:02d}_syn.npy'
    if os.path.exists(syn_path):
        syn_ts = np.load(syn_path)
        if len(syn_ts) >= MIX_LENGTH:
            print(f"  Loading cached synthetic series for run {run_idx}")
            syn_ts = syn_ts[:MIX_LENGTH]
        else:
            syn_ts = None
    else:
        syn_ts = None

    if syn_ts is None:
        print(f"  Generating fresh {MIX_LENGTH}-day synthetic series for run {run_idx}...")
        t0 = time.time()
        syn_ts = generate_timeseries_timegan(
            T=MIX_LENGTH, model=model, args=args,
            p_init=float(p_anchor), ret_mean=ret_mean, ret_std=ret_std,
            device=device, seed=seed_base + run_idx)
        np.save(syn_path, syn_ts)
        print(f"  Generation: {(time.time()-t0)/60:.1f} min")

    for prop in proportions:
        if prop in results:
            print(f"  Run {run_idx:2d} | Prop {prop:3d}% | already done — MAPE={results[prop]:.5f}")
            continue
        syn_len  = int(MIX_LENGTH * prop / 100)
        real_len = MIX_LENGTH - syn_len
        mix_dataset = build_mix_dataset(real_ts, syn_ts, real_len, syn_len, scaler)
        t0 = time.time()
        model_lstm, train_loss = train_lstm_full_batch(mix_dataset, seed=seed_base + run_idx + prop)
        train_time = time.time() - t0
        mape = evaluate_lstm(model_lstm, test_dataset, scaler)
        results[prop] = mape
        print(f"  Run {run_idx:2d} | Prop {prop:3d}% | windows={mix_dataset.shape[0]:5d} | "
              f"train_loss={train_loss:.5f} | MAPE={mape:.5f} | LSTM={train_time:.1f}s")
        sorted_props = sorted(results.keys())
        pd.DataFrame({'proportion_pct': sorted_props,
                      'mape': [results[p] for p in sorted_props]}).to_csv(run_csv, index=False)
        del model_lstm, mix_dataset
        torch.cuda.empty_cache()

    return {p: results[p] for p in proportions if p in results}

print("✓ TMTR-TimeGAN runner ready")


In [ ]:
# === MAIN LOOP ===
%matplotlib inline
from IPython.display import display, clear_output

all_results = {}
running_mape = np.full((N_RUNS, len(PROPORTIONS)), np.nan)

print(f"Pre-loading existing runs from {RES_DIR}...")
for run_idx in range(N_RUNS):
    run_csv = f'{RES_DIR}/run_{run_idx:02d}.csv'
    if os.path.exists(run_csv):
        df = pd.read_csv(run_csv)
        for j, p in enumerate(PROPORTIONS):
            mask = df['proportion_pct'] == p
            if mask.any():
                running_mape[run_idx, j] = df.loc[mask, 'mape'].values[0]
        all_results[run_idx] = dict(zip(df['proportion_pct'], df['mape']))
n_preloaded = int(sum(~np.all(np.isnan(running_mape), axis=1)))
print(f"  ✓ Found {n_preloaded} existing runs\n")


def plot_live(elapsed_min=None):
    valid = ~np.all(np.isnan(running_mape), axis=1)
    n_done = int(valid.sum())
    if n_done == 0:
        return
    clear_output(wait=True)
    fig, ax = plt.subplots()
    if n_done >= 2:
        n_iters = running_mape.shape[1]
        avg = np.full(n_iters, np.nan); mn = np.full(n_iters, np.nan); mx = np.full(n_iters, np.nan)
        for j in range(n_iters):
            res = running_mape[valid, j]
            res = np.sort(res[~np.isnan(res)])
            if len(res) < 2: continue
            pencentile = max(int(np.ceil(len(res) * 0.025)), 1)
            avg[j] = np.mean(res[pencentile:-pencentile]) if len(res) > 2*pencentile else np.mean(res)
            mn[j] = res[pencentile] if pencentile < len(res) else res[0]
            mx[j] = res[-pencentile] if pencentile < len(res) else res[-1]
        ax.plot(PROPORTIONS, avg, color='tab:red')
        ax.fill_between(PROPORTIONS, mn, mx, alpha=0.2, color='tab:red')
        if not np.isnan(avg[0]):
            ax.axhline(y=avg[0], color='gray', linestyle='--')
    else:
        ax.plot(PROPORTIONS, running_mape[0], color='tab:red')
    ax.set_xlabel('Syn. Prop. (%)'); ax.set_ylabel('MAPE')
    title = f'TMTR — {PRETTY_NAME} TimeGAN — {n_done}/{N_RUNS} runs'
    if elapsed_min is not None: title += f' (last: {elapsed_min:.1f} min)'
    ax.set_title(title, fontsize=10)
    plt.tight_layout()
    fig.savefig(f'{FIG_DIR}/live.png', dpi=120, bbox_inches='tight')
    display(fig); plt.close(fig)


todo_runs = [r for r in range(N_RUNS) if not (
    np.all(~np.isnan(running_mape[r])) and len(all_results.get(r, {})) >= len(PROPORTIONS)
)]
if not todo_runs:
    print(f"✓ All {N_RUNS} runs already complete.")
else:
    print(f"▶ Running {len(todo_runs)} pending runs ({todo_runs})\n")
    for run_idx in todo_runs:
        t0 = time.time()
        print(f"\n=== Run {run_idx+1}/{N_RUNS} ===")
        run_res = run_single_tmtr_timegan(run_idx, PROPORTIONS, seed_base=42)
        for j, p in enumerate(PROPORTIONS):
            if p in run_res:
                running_mape[run_idx, j] = run_res[p]
        all_results[run_idx] = run_res
        elapsed = (time.time() - t0) / 60
        plot_live(elapsed_min=elapsed)
        print(f"=== Run {run_idx+1}/{N_RUNS} done in {elapsed:.1f} min ===\n")

print(f"\n✓ Loop complete. Per-run CSVs in {RES_DIR}")


## 7. Phase 5 — Aggregation


In [ ]:
# Reload all per-run CSVs
rows = []
for run_idx in range(N_RUNS):
    p = f'{RES_DIR}/run_{run_idx:02d}.csv'
    if not os.path.exists(p):
        continue
    df = pd.read_csv(p)
    df['run_idx'] = run_idx
    rows.append(df)

df_all = pd.concat(rows, ignore_index=True)
df_pivot = df_all.pivot(index='run_idx', columns='proportion_pct', values='mape')
print(f"Results matrix: {df_pivot.shape}")
print(df_pivot.round(5))


def summarize_authors_style(errors_matrix):
    n_runs, n_iters = errors_matrix.shape
    summary = np.zeros((3, n_iters))
    for i in range(n_iters):
        res = errors_matrix[:, i].copy()
        res.sort()
        pencentile = max(int(np.ceil(len(res) * 0.025)), 1)
        summary[0, i] = np.mean(res[pencentile:-pencentile]) if len(res) > 2*pencentile else np.mean(res)
        summary[1, i] = res[pencentile]
        summary[2, i] = res[-pencentile]
    return pd.DataFrame({'avg': summary[0, :], 'min': summary[1, :], 'max': summary[2, :]})


errors_matrix = df_pivot[PROPORTIONS].to_numpy()
summary_df_authors = summarize_authors_style(errors_matrix)
summary_df_authors['proportion_pct'] = PROPORTIONS
summary_df_authors = summary_df_authors[['proportion_pct', 'avg', 'min', 'max']]
summary_df_authors['n_runs'] = (~np.isnan(errors_matrix)).sum(axis=0)
summary_df_authors.to_csv(f'{RES_DIR}/summary_authors_style.csv', index=False)
df_pivot.to_csv(f'{RES_DIR}/results_matrix.csv')

def bootstrap_ci(x, n_boot=10000, ci=0.95, seed=0):
    rng = np.random.RandomState(seed)
    n = len(x)
    boots = np.array([rng.choice(x, size=n, replace=True).mean() for _ in range(n_boot)])
    return np.quantile(boots, [(1-ci)/2, 1-(1-ci)/2])

summary_stats = []
for s in PROPORTIONS:
    vals = df_pivot[s].dropna().values
    if len(vals) == 0:
        continue
    lo, hi = bootstrap_ci(vals)
    summary_stats.append({'proportion_pct': s, 'mean_mape': vals.mean(), 'std_mape': vals.std(),
                          'ci95_low': lo, 'ci95_high': hi, 'n_runs': len(vals)})
summary_df = pd.DataFrame(summary_stats)
summary_df.to_csv(f'{RES_DIR}/summary.csv', index=False)
print("✓ Saved summaries.")


## 8. Phase 6 — Final Figures


In [ ]:
# === Final figures: paper-faithful + enhanced ===

# --- Authors' exact style ---
fig, ax = plt.subplots()
x_range = np.array(PROPORTIONS)
ax.plot(x_range, summary_df_authors['avg'].values, color='tab:red', label='TimeGAN')
ax.fill_between(x_range, summary_df_authors['min'].values, summary_df_authors['max'].values,
                alpha=0.2, color='tab:red')
ax.axhline(y=summary_df_authors['avg'].values[0], color='gray', linestyle='--')
ax.set_xlabel('Syn. Prop. (%)'); ax.set_ylabel('MAPE')
plt.tight_layout()
fig.savefig(f'{FIG_DIR}/final.pdf', bbox_inches='tight')
fig.savefig(f'{FIG_DIR}/final.png', bbox_inches='tight', dpi=150)
plt.show(); plt.close(fig)
print(f"✓ Authors-style figure saved to {FIG_DIR}/final.pdf")

# --- Enhanced style ---
fig2, ax2 = plt.subplots(figsize=(11, 6))
mean = summary_df['mean_mape'].values
lo   = summary_df['ci95_low'].values
hi   = summary_df['ci95_high'].values
xv   = summary_df['proportion_pct'].values
n_runs_actual = int(summary_df['n_runs'].iloc[0])

ax2.plot(xv, mean, 'o-', linewidth=2.5, color='tab:red', markersize=7,
         label=f'TimeGAN (n={n_runs_actual} runs)')
ax2.fill_between(xv, lo, hi, alpha=0.25, color='tab:red', label='95% bootstrap CI')
ax2.axhline(mean[0], color='gray', linestyle='--', alpha=0.7,
            label=f'Baseline = {mean[0]:.4f}')
for x, y in zip(xv, mean):
    if np.isnan(y): continue
    ax2.annotate(f'{y:.4f}', xy=(x, y), xytext=(0, 12),
                 textcoords='offset points', ha='center', fontsize=9,
                 fontweight='bold', color='darkred',
                 bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='darkred', alpha=0.85))

ax2.set_xlabel('Syn. Prop. (%)', fontsize=12)
ax2.set_ylabel('MAPE (real test set)', fontsize=12)
ax2.set_title(f'TMTR — {PRETTY_NAME} TimeGAN — {n_runs_actual} runs',
              fontsize=12, fontweight='bold')
ax2.legend(loc='upper right', fontsize=10); ax2.grid(alpha=0.3)
plt.tight_layout()
fig2.savefig(f'{FIG_DIR}/final_enhanced.pdf', bbox_inches='tight', dpi=150)
fig2.savefig(f'{FIG_DIR}/final_enhanced.png', bbox_inches='tight', dpi=150)
plt.show(); plt.close(fig2)
print(f"✓ Enhanced figure saved to {FIG_DIR}/final_enhanced.pdf")


## 9. Cross-generator comparison: FTS-Diffusion vs TimeGAN

Loads the corresponding FTS-Diffusion summary (if present on Drive) and
overlays both curves on a single figure. Run the FTS-Diffusion
TMTR notebook for the same `(ASSET, K, PROTOCOL)` first if you want
the comparison.


In [ ]:
# === Cross-generator comparison plot ===
COMPARE_K        = 14         # which K to compare against (FTS-Diffusion specific)
COMPARE_PROTOCOL = 'authors'  # which FTS-Diffusion protocol to compare against

fts_summary_path     = f'{PERSIST_ROOT}/results/tmtr/{ASSET}/k{COMPARE_K}/{COMPARE_PROTOCOL}/summary.csv'
timegan_summary_path = f'{RES_DIR}/summary.csv'

curves = {}
if os.path.exists(fts_summary_path):
    curves[f'FTS-Diffusion (K={COMPARE_K}, {COMPARE_PROTOCOL})'] = pd.read_csv(fts_summary_path)
    print(f"  ✓ FTS-Diffusion: {fts_summary_path}")
else:
    print(f"  — FTS-Diffusion summary not yet available at {fts_summary_path}")

if os.path.exists(timegan_summary_path):
    curves['TimeGAN'] = pd.read_csv(timegan_summary_path)
    print(f"  ✓ TimeGAN: {timegan_summary_path}")

if len(curves) >= 2:
    colors = {f'FTS-Diffusion (K={COMPARE_K}, {COMPARE_PROTOCOL})': 'steelblue',
              'TimeGAN': 'tab:red'}
    fig, ax = plt.subplots(figsize=(11, 6))
    for label, df_p in curves.items():
        c = colors.get(label, 'gray')
        x = df_p['proportion_pct']
        ax.plot(x, df_p['mean_mape'], 'o-', color=c, linewidth=2, markersize=6,
                label=f"{label} (n={int(df_p['n_runs'].iloc[0])})")
        ax.fill_between(x, df_p['ci95_low'], df_p['ci95_high'], color=c, alpha=0.2)

    ref = list(curves.values())[0]
    ax.axhline(ref['mean_mape'].iloc[0], color='gray', linestyle='--', alpha=0.6,
               label=f"Baseline = {ref['mean_mape'].iloc[0]:.4f}")

    ax.set_xlabel('Synthetic Proportion (%)', fontsize=12)
    ax.set_ylabel('MAPE (real test set)', fontsize=12)
    ax.set_title(f'TMTR cross-generator comparison — {PRETTY_NAME}',
                 fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=10); ax.grid(alpha=0.3)
    plt.tight_layout()

    cmp_dir = f'{PERSIST_ROOT}/figures/tmtr/{ASSET}/_cross_generator'
    os.makedirs(cmp_dir, exist_ok=True)
    fig.savefig(f'{cmp_dir}/k{COMPARE_K}_{COMPARE_PROTOCOL}_vs_timegan.pdf', bbox_inches='tight')
    fig.savefig(f'{cmp_dir}/k{COMPARE_K}_{COMPARE_PROTOCOL}_vs_timegan.png', bbox_inches='tight', dpi=150)
    plt.show(); plt.close(fig)
    print(f"\n✓ Cross-generator figure saved to {cmp_dir}/")
else:
    print(f"\n⚠ Need both FTS-Diffusion and TimeGAN summaries to overlay.")


## 10. How to resume / switch asset

### Resume an interrupted run
Just re-execute the main loop (Phase 4). Pre-loads existing CSVs and only
runs the missing replicates / sweep points.

### Switch asset
Edit cell 6 (Configuration): change `ASSET`. Re-run from Phase 1; if the
TimeGAN checkpoint for the new asset doesn't exist on Drive yet, training
re-runs (~30-60 min on A100).

### Force-retrain TimeGAN
Delete `{ARCH_DIR}/model.pt` and `{ARCH_DIR}/args.pickle` from Drive,
then re-run Phase 2.

### Compare protocols
This notebook has a single TimeGAN protocol. The comparison axis is
**generator** (FTS-Diffusion vs TimeGAN), driven by Phase 9.
